# Canonical 128px rescore of the 1,000-pair set; Gaussian-W2 row; figure candidates

Rescoring pass at `SCORE_RES=128` that produces Table 1's `tab:canonical` CNT block (cell 2) and the Gaussian row (cell 10, printed only -- FID/KID re-verified from `matching_ablation/feat_gaussian_w2_inc.npy`). Also writes figure candidates under `paper_figs/{grid,boundary,failure,apps}`.

**Needs:** the rendered images from `benchmark_pairs_and_rendering.ipynb`, the canonical 2,819-image reference.

**Maintained runnable path:** `score_only.py` recomputes every value this notebook produces, from the same cached features, with no rendering. Kept for provenance.

# Finish day — run everything, generate figure candidates
Order: setup → 128px rescore (closes the protocol) → figure candidates, many per slot, saved to `DATA_ROOT/paper_figs/<slot>/`.
Pick one candidate per slot and copy it to `figs/` under the paper's expected name (printed at the end).
Resumable throughout; "Run all" is safe after a disconnect.

In [ ]:
# ============================================================
# 0. FAST SETUP
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'; os.makedirs(OUT,exist_ok=True)
FIGS=f'{OUT}/figs'; os.makedirs(FIGS,exist_ok=True); TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
NUM={}; note=lambda k,v:(NUM.__setitem__(k,v),print(f'  {k} = {v}'))[0]

t0=time.time()
# --- DTD via single zip (created once, then reused forever) ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    if os.path.exists(ZIP) and os.path.getsize(ZIP)>500e6:
        os.system(f'unzip -qo "{ZIP}" -d /content/_dtd && rm -rf {LOCAL} && mv /content/_dtd/images {LOCAL}')
    else:
        if os.path.exists(ZIP): os.remove(ZIP)   # partial zip from an interrupted run
        # stream Drive->local with tar (sequential read beats per-file cp/zip on the FUSE mount),
        # then zip LOCALLY and move ONE file to Drive for future sessions
        os.system(f'cd "{BASE}/dtd" && tar cf - images | (cd /content && tar xf -) && mv /content/images {LOCAL}')
        if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:   # Drive throttled -> fetch from source instead
            os.system('wget -q https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz -O /content/dtd.tar.gz')
            os.system(f'cd /content && tar xzf dtd.tar.gz dtd/images && rm -rf {LOCAL} && mv dtd/images {LOCAL}')
        os.system(f'cd /content && zip -qr /content/dtd_local.zip dtd_images && mv /content/dtd_local.zip "{ZIP}"')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')

# --- minimal pip (torch/scipy/sklearn preinstalled on Colab) ---
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
print(f'model ready {time.time()-t0:.0f}s')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def _dec(d,it,L,H,W,Cc,layout='a'):
    it=it.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_"+layout],{s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def interp_exact(d,eta,seed,n_exact=1024):
    L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t'); src,tgt=sf[i_s],tf[i_t]
    g=torch.Generator(device=device).manual_seed(seed+11)
    js=torch.randperm(NSUB,generator=g,device=device)[:n_exact]; is_=torch.randperm(NSUB,generator=g,device=device)[:n_exact]
    r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
    tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
    ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,Cc)

# --- observers, BATCHED ---
import timm
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)

ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def fid(X,Y):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False)
    cm,_=linalg.sqrtm(Sx@Sy,disp=False); cm=cm.real if np.iscomplexobj(cm) else cm
    return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
print(f'setup done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# 1. 128px RESCORE — canonical protocol: clear 256px caches, rescore anchors + all methods + endpoint gap at 128px
# ============================================================
import glob as _g
for f in _g.glob(f'{OUT}/feat_*_inc.npy')+_g.glob(f'{OUT}/feat_*_dino.npy')+[f'{OUT}/anchor_dino.npz',f'{OUT}/exact_endgap.npy']:
    if os.path.exists(f): os.remove(f); print('rm',os.path.basename(f))
SCORE_RES=128
def _open128(f): return Image.open(f).convert('RGB').resize((SCORE_RES,SCORE_RES))

ANC=f'{OUT}/anchor_dino.npz'
anc={}; bA=[];bB=[];kk=[]; t0=time.time()
for i,p_ in enumerate(MAN_ALL):
    k=int(p_['k'])
    bA.append(to_pil(make_recon(Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'))).resize((SCORE_RES,SCORE_RES)))
    bB.append(to_pil(make_recon(Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'))).resize((SCORE_RES,SCORE_RES))); kk.append(k)
    if len(kk)==64 or i==len(MAN_ALL)-1:
        fA=dino_feats(bA); fB=dino_feats(bB)
        for j,k2 in enumerate(kk): anc[f'{k2}_A']=fA[j]; anc[f'{k2}_B']=fB[j]
        bA,bB,kk=[],[],[]
        if i%256<64: print(f'anchors {i+1}/1000 {time.time()-t0:.0f}s')
np.savez(ANC,**anc); print('anchors done')

METHODS={m:p for m,p in {'exact':f'{SCALE}/exact','ot':f'{SCALE}/ot','ot_sym':f'{SCALE}/ot_sym','gaussian_ot':f'{SCALE}/gaussian_ot',
         'pixel_linear':f'{SCALE}/pixel_linear','texton_linear':f'{SCALE}/texton_linear','tm':f'{SCALE}/mid'}.items() if os.path.isdir(p)}
rows=[]; XIs={}
for m,pdir in METHODS.items():
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'
    fs=[f'{pdir}/{int(p_["k"]):04d}.png' for p_ in MAN_ALL]; fs=[f for f in fs if os.path.exists(f)]
    ims=[_open128(f) for f in fs]; XI=inc_feats(ims); XD=dino_feats(ims); np.save(fI,XI); np.save(fD,XD); XIs[m]=XI
    ks=[int(os.path.basename(f)[:4]) for f in fs]
    rI=realism(XI,MANI_INC); rD=realism(XD,MANI_DINO)
    plc=np.array([np.linalg.norm(XD[i]-anc[f'{k}_A'])/(np.linalg.norm(XD[i]-anc[f'{k}_A'])+np.linalg.norm(XD[i]-anc[f'{k}_B'])+1e-9) for i,k in enumerate(ks)])
    for i,k in enumerate(ks): rows.append(dict(k=k,method=m,real_inc=rI[i],real_dino=rD[i],place=plc[i]))
    print(m,'scored',len(ks))
sc=pd.DataFrame(rows); sc.to_csv(f'{OUT}/final128_n1000.csv',index=False)
ims=[_open128(f'{SCALE}/exact/{int(p_["k"]):04d}_end.png') for p_ in MAN_ALL]
XE=dino_feats(ims)
eg=np.array([np.linalg.norm(XE[i]-anc[f'{int(p_["k"])}_B'])/(np.linalg.norm(anc[f'{int(p_["k"])}_A']-anc[f'{int(p_["k"])}_B'])+1e-9) for i,p_ in enumerate(MAN_ALL)])
np.save(f'{OUT}/exact_endgap.npy',eg)

piv={c:sc.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
print(f"\n=== CANONICAL 128px, n=1000 ===")
print(f"{'method':14}{'FID':>8}{'KID':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}")
res128={}
for m in METHODS:
    Fv=fid(XIs[m],ref_inc); Kv=kid(XIs[m],ref_inc)
    res128[m]=dict(FID=round(Fv,1),KID=round(Kv[0],4),RInc=round(float(piv['real_inc'][m].median()),3),
                   RDin=round(float(piv['real_dino'][m].median()),3),Place=round(float((piv['place'][m]-.5).abs().median()),3))
    print(f"{m:14}{Fv:8.1f}{Kv[0]:9.4f}±{Kv[1]:.4f}{res128[m]['RInc']:9.3f}{res128[m]['RDin']:10.3f}{res128[m]['Place']:11.3f}")
print('exact endpoint gap:',round(float(np.median(eg)),3))
json.dump(res128,open(f'{OUT}/numbers_final128.json','w'),indent=1)
print('\n>>> paste this whole table back to Claude for numbers.tex <<<')

In [ ]:
# ============================================================
# 2. SHARED FIGURE HELPERS — sinkhorn/OT/gauss operators + save-to-candidates
# ============================================================
PF=f'{BASE}/paper_figs'; os.makedirs(PF,exist_ok=True)
def cand_dir(slot):
    d=f'{PF}/{slot}'; os.makedirs(d,exist_ok=True); return d
def sinkhorn_plan(x,y,eps,iters=100):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def interp_ot(d,eta,seed=0,eps=EPS):
    L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    P,_=sinkhorn_plan(sf[i_s],tf[i_t],eps); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,Cc)
def _sym(M): return .5*(M+M.T)
def _psd_sqrt(M,fl):
    e,V=torch.linalg.eigh(_sym(M)); e=e.clamp_min(fl); return (V*e.sqrt())@V.T
def _psd_invsqrt(M,fl):
    e,V=torch.linalg.eigh(_sym(M)); e=e.clamp_min(fl); return (V*e.rsqrt())@V.T
def interp_gauss(d,eta,seed=0,ridge=1e-4):
    L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    X,Y=sf[i_s].float(),tf[i_t].float(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx
    I=torch.eye(Cc,device=sf.device)
    Sa=_sym((Xc.T@Xc)/(len(X)-1))+ridge*I; Yc=Y-my; Sb=_sym((Yc.T@Yc)/(len(Y)-1))+ridge*I
    Sh=_psd_sqrt(Sa,ridge); Sih=_psd_invsqrt(Sa,ridge)
    A=_sym(Sih@_psd_sqrt(_sym(Sh@Sb@Sh),0.0)@Sih)
    tt=my+Xc@A.T; nn=torch.cdist(sf,sf[i_s]).argmin(1)
    return _dec(d,(1-eta)*sf.float()+eta*tt[nn],L,H,W,Cc)
def interp_lin_texton(d,eta):
    bf=(1-eta)*d["bf_a"]+eta*d["bf_b"]; si=(1-eta)*d["si_a"]+eta*d["si_b"]
    sm=build_style_maps(bf,si,FMS); return decode_with_style_maps(d["blob_a"],sm)
def pair_imgs(k):
    return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
print('helpers ready; candidates go to',PF)

In [ ]:
# ============================================================
# 3. METHOD FIGURE candidates — 4 real-data panels per candidate pair -> paper_figs/method/
# ============================================================
from sklearn.decomposition import PCA
CAND_KS=[3,17,42,88,140,205,333]     # edit freely; each produces one candidate
D=cand_dir('method')
for k in CAND_KS:
    try:
        A,B=pair_imgs(k); d=build_d(cnt_load(A),cnt_load(B)); L,Cc,H,W,sf,tf=_grids(d)
        i_s=_sub(sf.shape[0],k,'s'); i_t=_sub(tf.shape[0],k,'t')
        Xs=sf[i_s][:30].cpu().numpy(); Ys=tf[i_t][:30].cpu().numpy()
        P,_=sinkhorn_plan(sf[i_s][:30],tf[i_t][:30],EPS)
        Pn=(P/(P.sum(1,keepdim=True)+1e-10)).cpu().numpy()
        pca=PCA(2).fit(np.concatenate([Xs,Ys])); Xp,Yp=pca.transform(Xs),pca.transform(Ys)
        # style-map PCA->RGB
        def smap_rgb(cloud):
            p3=PCA(3).fit_transform(cloud.cpu().numpy()); p3=(p3-p3.min(0))/(p3.ptp(0)+1e-9)
            return p3.reshape(H,W,3)
        fig=plt.figure(figsize=(16,4.2)); gs=fig.add_gridspec(2,8,height_ratios=[1,1])
        ax=fig.add_subplot(gs[0,0]); ax.imshow(A.resize((256,256))); ax.set_title('A'); ax.axis('off')
        ax=fig.add_subplot(gs[1,0]); ax.imshow(B.resize((256,256))); ax.set_title('B'); ax.axis('off')
        ax=fig.add_subplot(gs[0,1]); ax.imshow(smap_rgb(sf)); ax.set_title('style map A (PCA)',fontsize=8); ax.axis('off')
        ax=fig.add_subplot(gs[1,1]); ax.imshow(smap_rgb(tf)); ax.set_title('style map B (PCA)',fontsize=8); ax.axis('off')
        ax=fig.add_subplot(gs[:,2:4])
        ax.scatter(*Xp.T,c='tab:blue',s=22,label='A features'); ax.scatter(*Yp.T,c='tab:orange',s=22,label='B features')
        for i in range(30): ax.plot([Xp[i,0],Yp[i,0]],[Xp[i,1],Yp[i,1]],c='grey',alpha=.25,lw=.8)
        ax.set_title('positional pairing',fontsize=9); ax.legend(fontsize=7); ax.set_xticks([]); ax.set_yticks([])
        ax=fig.add_subplot(gs[:,4:6])
        ax.scatter(*Xp.T,c='tab:blue',s=22); ax.scatter(*Yp.T,c='tab:orange',s=22)
        for i in range(30):
            for j in np.argsort(-Pn[i])[:3]:
                ax.plot([Xp[i,0],Yp[j,0]],[Xp[i,1],Yp[j,1]],c='tab:green',alpha=float(min(1,Pn[i,j]*3)),lw=1.1)
        ax.set_title('OT coupling (soft plan drawn soft)',fontsize=9); ax.set_xticks([]); ax.set_yticks([])
        for j,e in enumerate([0,1/3,2/3,1.0]):
            ax=fig.add_subplot(gs[j//2, 6+j%2]); ax.imshow(to_pil(interp_ot(d,e,seed=k))); ax.set_title(f'η={e:.2f}',fontsize=8); ax.axis('off')
        plt.suptitle(f'method figure candidate — pair {k}',fontsize=10)
        plt.tight_layout(); plt.savefig(f'{D}/pair{k:04d}.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
    except Exception as e: print('pair',k,'failed:',e)
print('method candidates in',D,'-> chosen one becomes figs/method_figure.png')

In [ ]:
# ============================================================
# 4. MAIN GRID candidates — 3 pair-selections x [A|CNT-linear|GaussW2|OT|(Vacher)|TM|B] -> paper_figs/grid/
# ============================================================
D=cand_dir('grid')
VAC=f'{SCALE}/vacher'   # include column if you have Vacher midpoints saved here
SETS={'setA':[12,145,317,502,688,901],'setB':[45,210,388,555,760,955],'setC':[80,260,430,610,808,990]}
cols=[('A',f'{SCALE}/A'),('CNT-linear',f'{SCALE}/texton_linear'),('Gaussian $W_2$',f'{SCALE}/gaussian_ot'),('OT',f'{SCALE}/ot')]
if os.path.isdir(VAC): cols.append(('Vacher-$W_2$',VAC))
cols+= [('Texture Mixer',f'{SCALE}/mid'),('B',f'{SCALE}/B')]
for name,ks in SETS.items():
    ks=[k for k in ks if all(os.path.exists(f'{p}/{k:04d}.png') for _,p in cols)]
    fig,ax=plt.subplots(len(ks),len(cols),figsize=(1.9*len(cols),1.9*len(ks)))
    for i,k in enumerate(ks):
        for j,(t,p) in enumerate(cols):
            ax[i,j].imshow(Image.open(f'{p}/{k:04d}.png').convert('RGB').resize((256,256))); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(t,fontsize=9)
    plt.tight_layout(); plt.savefig(f'{D}/{name}.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('grid candidates in',D,'-> chosen becomes figs/grid_main.png','| Vacher column included:',os.path.isdir(VAC))

In [ ]:
# ============================================================
# 5. BOUNDARY FIGURE candidates — rows: small warp / overlapping crops / dissimilar / (faces if provided); SIFT counts -> paper_figs/boundary/
# ============================================================
import cv2
D=cand_dir('boundary'); FACES=f'{BASE}/faces'   # optional: put two aligned face crops here as face_A.png/face_B.png
def sift_count(a,b):
    s=cv2.SIFT_create(); g=lambda im: cv2.cvtColor(np.asarray(im.resize((256,256))),cv2.COLOR_RGB2GRAY)
    ka,da=s.detectAndCompute(g(a),None); kb,db=s.detectAndCompute(g(b),None)
    if da is None or db is None: return 0
    m=cv2.BFMatcher().knnMatch(da,db,k=2)
    return sum(1 for p in m if len(p)==2 and p[0].distance<0.75*p[1].distance)
def row_from(A,B,tag):
    d=build_d(cnt_load(A),cnt_load(B))
    lin=to_pil(interp_lin_texton(d,0.5)); ot=to_pil(interp_ot(d,0.5,seed=1))
    return [A.resize((256,256)),lin,ot,B.resize((256,256))],sift_count(A,B),tag
BASE_KS=[12,145,317]     # 3 candidates: base texture per candidate
for k in BASE_KS:
    A,B=pair_imgs(k); W0,H0=A.size
    rows=[]
    warp=A.rotate(6,resample=Image.BICUBIC).crop((20,20,W0-20,H0-20)).resize(A.size)
    rows.append(row_from(A,warp,'small warp'))
    big=A.resize((int(W0*1.4),int(H0*1.4)))
    c1=big.crop((0,0,W0,H0)); c2=big.crop((int(W0*.25),int(H0*.25),int(W0*.25)+W0,int(H0*.25)+H0))
    rows.append(row_from(c1,c2,'overlapping crops'))
    rows.append(row_from(A,B,'dissimilar textures'))
    if os.path.exists(f'{FACES}/face_A.png'):
        rows.append(row_from(Image.open(f'{FACES}/face_A.png').convert('RGB'),Image.open(f'{FACES}/face_B.png').convert('RGB'),'aligned faces'))
    fig,ax=plt.subplots(len(rows),4,figsize=(9,2.3*len(rows)))
    for i,(ims,cnt,tag) in enumerate(rows):
        for j,im in enumerate(ims): ax[i,j].imshow(im); ax[i,j].axis('off')
        ax[i,0].set_ylabel(f'{tag}\nSIFT={cnt}',fontsize=8,rotation=0,ha='right',va='center',labelpad=45)
        if i==0:
            for j,t in enumerate(['A','linear','OT','B']): ax[i,j].set_title(t,fontsize=9)
    plt.tight_layout(); plt.savefig(f'{D}/base{k:04d}.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('boundary candidates in',D,'-> chosen becomes figs/boundary.png',
      '| faces row:',os.path.exists(f'{FACES}/face_A.png'))

In [ ]:
# ============================================================
# 6. FAILURE PANEL candidates — periodic / rho*-flagged / high-correspondence -> paper_figs/failure/
# ============================================================
D=cand_dir('failure')
FLAGGED=[649,747,27,631,179,447]      # from the rho* run
PERIODIC=sorted(glob.glob(f'{LOCAL}/chequered/*.jpg'))[:3]+sorted(glob.glob(f'{LOCAL}/grid/*.jpg'))[:3]
for v,(per,fk) in enumerate([(PERIODIC[0],FLAGGED[0]),(PERIODIC[1],FLAGGED[1]),(PERIODIC[3],FLAGGED[2])]):
    fig,ax=plt.subplots(3,4,figsize=(9,7))
    Ap=Image.open(per).convert('RGB'); Bp=Image.open(PERIODIC[2]).convert('RGB')
    d=build_d(cnt_load(Ap),cnt_load(Bp))
    for j,(t,im) in enumerate([('A',Ap.resize((256,256))),('linear',to_pil(interp_lin_texton(d,.5))),('OT',to_pil(interp_ot(d,.5,seed=2))),('B',Bp.resize((256,256)))]):
        ax[0,j].imshow(im); ax[0,j].set_title(t,fontsize=9); ax[0,j].axis('off')
    ax[0,0].set_ylabel('periodic',fontsize=9)
    A2,B2=pair_imgs(fk); d2=build_d(cnt_load(A2),cnt_load(B2))
    for j,im in enumerate([A2.resize((256,256)),to_pil(interp_lin_texton(d2,.5)),to_pil(interp_ot(d2,.5,seed=2)),B2.resize((256,256))]):
        ax[1,j].imshow(im); ax[1,j].axis('off')
    ax[1,0].set_ylabel(f'$\\rho^\\star$-flagged (k={fk})',fontsize=9)
    A3,_=pair_imgs(12); W0,H0=A3.size; big=A3.resize((int(W0*1.3),int(H0*1.3)))
    c1=big.crop((0,0,W0,H0)); c2=big.crop((int(W0*.15),int(H0*.15),int(W0*.15)+W0,int(H0*.15)+H0))
    d3=build_d(cnt_load(c1),cnt_load(c2))
    for j,im in enumerate([c1.resize((256,256)),to_pil(interp_lin_texton(d3,.5)),to_pil(interp_ot(d3,.5,seed=2)),c2.resize((256,256))]):
        ax[2,j].imshow(im); ax[2,j].axis('off')
    ax[2,0].set_ylabel('high correspondence',fontsize=9)
    plt.tight_layout(); plt.savefig(f'{D}/variant{v}.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('failure candidates in',D,'-> chosen becomes figs/failure_panel.png')

In [ ]:
# ============================================================
# 7. APPS COMPOSITE candidates — transfer / spatial eta ramp / 4-way -> paper_figs/apps/
# ============================================================
D=cand_dir('apps')
hits=glob.glob(f'{BASE}/**/4_way_barrycenter_2.png',recursive=True)
FOURWAY=hits[0] if hits else f'{BASE}/4_way_barrycenter_2.png'
CAND=[ (12,317), (145,688), (45,555) ]
def spatial_eta(d):
    L,Cc,H,W,sf,tf=_grids(d); i_s=_sub(sf.shape[0],3,'s'); i_t=_sub(tf.shape[0],3,'t')
    P,_=sinkhorn_plan(sf[i_s],tf[i_t],EPS); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); ramp=torch.linspace(0,1,W,device=device).repeat(H,1).reshape(-1,1)
    return _dec(d,(1-ramp)*sf+ramp*tt[nn],L,H,W,Cc)
for v,(ka,kb) in enumerate(CAND):
    A,B=pair_imgs(ka); d=build_d(cnt_load(A),cnt_load(B))
    tr_ab=to_pil(interp_ot(d,1.0,seed=ka))                       # A layout, B material
    d_ba=build_d(cnt_load(B),cnt_load(A)); tr_ba=to_pil(interp_ot(d_ba,1.0,seed=ka))
    sp=to_pil(spatial_eta(d))
    fig,ax=plt.subplots(1,6,figsize=(16,2.8))
    for j,(t,im) in enumerate([('A',A.resize((256,256))),("A layout, B material",tr_ab),("B layout, A material",tr_ba),('B',B.resize((256,256))),('spatial η(x) ramp',sp),('anchor-based 4-way','FOURWAY')]):
        if im=='FOURWAY':
            if os.path.exists(FOURWAY):
                fw=Image.open(FOURWAY).convert('RGB'); w,h=fw.size; fw=fw.crop((0,int(h*0.12),w,h))  # crop title band
                ax[j].imshow(fw)
            else: ax[j].text(.5,.5,'4-way PNG\nnot found',ha='center')
        else: ax[j].imshow(im)
        ax[j].set_title(t,fontsize=8); ax[j].axis('off')
    plt.tight_layout(); plt.savefig(f'{D}/variant{v}.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('apps candidates in',D,'-> chosen becomes figs/apps_row.png')

In [ ]:
# ============================================================
# 8. FINAL CHECKLIST — expected paper filenames vs chosen candidates
# ============================================================
need={'figs/transition.png':'have (teaser)','figs/method_figure.png':'choose from paper_figs/method/',
'figs/grid_main.png':'choose from paper_figs/grid/','figs/boundary.png':'choose from paper_figs/boundary/',
'figs/failure_panel.png':'choose from paper_figs/failure/','figs/apps_row.png':'choose from paper_figs/apps/'}
for k,v in need.items(): print(f'{k:28} <- {v}')
print('\nAfter choosing: copy each into the paper repo figs/ under the left-hand name.')
print('Then paste the CANONICAL 128px table (cell 1 output) + Vacher-150 number back to Claude.')

In [ ]:
# --- fix the Gaussian dir, rescore just gaussian, rebuild grid ---
print([p for p in os.listdir(SCALE) if 'gauss' in p.lower()])   # find where FINAL wrote images
GD=f'{SCALE}/gaussian_w2'                                       # canonical corrected dir
if not os.path.isdir(GD) or len(glob.glob(f'{GD}/*.png'))<1000:
    os.makedirs(GD,exist_ok=True)                               # regenerate with cell-2's interp_gauss (it IS the corrected map)
    todo=[int(p_['k']) for p_ in MAN_ALL if not os.path.exists(f'{GD}/{int(p_["k"]):04d}.png')]
    print(len(todo),'to generate'); t0=time.time()
    for n,k in enumerate(todo):
        A,B=pair_imgs(k); d=build_d(cnt_load(A),cnt_load(B))
        to_pil(interp_gauss(d,0.5,seed=k)).save(f'{GD}/{k:04d}.png')
        if n%200==0: print(n,f'{time.time()-t0:.0f}s')
ims=[Image.open(f'{GD}/{int(p_["k"]):04d}.png').convert('RGB').resize((128,128)) for p_ in MAN_ALL]
XI=inc_feats(ims); XD=dino_feats(ims)
np.save(f'{OUT}/feat_gaussian_w2_inc.npy',XI); np.save(f'{OUT}/feat_gaussian_w2_dino.npy',XD)
anc=dict(np.load(f'{OUT}/anchor_dino.npz'))
plc=np.array([np.linalg.norm(XD[i]-anc[f'{int(p_["k"])}_A'])/(np.linalg.norm(XD[i]-anc[f'{int(p_["k"])}_A'])+np.linalg.norm(XD[i]-anc[f'{int(p_["k"])}_B'])+1e-9) for i,p_ in enumerate(MAN_ALL)])
Fv=fid(XI,ref_inc); Kv=kid(XI,ref_inc)
print(f"gaussian_w2 CORRECTED @128: FID {Fv:.1f}  KID {Kv[0]:.4f}±{Kv[1]:.4f}  realInc {np.median(realism(XI,MANI_INC)):.3f}  realDINO {np.median(realism(XD,MANI_DINO)):.3f}  |place-.5| {np.median(np.abs(plc-.5)):.3f}")
# sanity: first image must look like a texture, not blobs
plt.imshow(ims[0]); plt.axis('off'); plt.show()

In [ ]:
D=cand_dir('boundary')
def sift_count(a,b):
    try:
        s=cv2.SIFT_create(); g=lambda im: cv2.cvtColor(np.asarray(im.resize((256,256))),cv2.COLOR_RGB2GRAY)
        ka,da=s.detectAndCompute(g(a),None); kb,db=s.detectAndCompute(g(b),None)
        if da is None or db is None: return 0
        return sum(1 for p in cv2.BFMatcher().knnMatch(da,db,k=2) if len(p)==2 and p[0].distance<0.75*p[1].distance)
    except Exception: return -1
for k in [12,145,317]:
    A,B=pair_imgs(k); W0,H0=A.size; rows=[]
    warp=A.rotate(6,resample=Image.BICUBIC).crop((20,20,W0-20,H0-20)).resize(A.size)
    big=A.resize((int(W0*1.4),int(H0*1.4)))
    c1=big.crop((0,0,W0,H0)); c2=big.crop((int(W0*.25),int(H0*.25),int(W0*.25)+W0,int(H0*.25)+H0))
    for A2,B2,tag in [(A,warp,'small warp'),(c1,c2,'overlapping crops'),(A,B,'dissimilar')]:
        d=build_d(cnt_load(A2),cnt_load(B2))
        rows.append(([A2.resize((256,256)),to_pil(interp_lin_texton(d,.5)),to_pil(interp_ot(d,.5,seed=1)),B2.resize((256,256))],sift_count(A2,B2),tag))
    fig,ax=plt.subplots(len(rows),4,figsize=(10,2.5*len(rows)))
    for i,(ims_,cnt,tag) in enumerate(rows):
        for j,im in enumerate(ims_): ax[i,j].imshow(im); ax[i,j].axis('off')
        ax[i,0].text(-0.08,0.5,f'{tag}\nSIFT={cnt}',transform=ax[i,0].transAxes,fontsize=9,ha='right',va='center')
        if i==0:
            for j,t in enumerate(['A','linear','OT','B']): ax[i,j].set_title(t,fontsize=10)
    plt.tight_layout(); plt.savefig(f'{D}/base{k:04d}_fixed.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()

In [ ]:
# GRID rebuild: A | CNT-linear | Gaussian W2 (corrected) | OT | (Vacher) | TM | B
D=cand_dir('grid')
VAC=f'{SCALE}/vacher'
SETS={'setA':[12,145,317,502,688,901],'setB':[45,210,388,555,760,955],'setC':[80,260,430,610,808,990]}
cols=[('A',f'{SCALE}/A'),('CNT-linear',f'{SCALE}/texton_linear'),('Gaussian $W_2$',f'{SCALE}/gaussian_w2'),('OT',f'{SCALE}/ot')]
if os.path.isdir(VAC) and len(glob.glob(f'{VAC}/*.png'))>0: cols.append(('Vacher-$W_2$',VAC))
cols += [('Texture Mixer',f'{SCALE}/mid'),('B',f'{SCALE}/B')]
for name,ks in SETS.items():
    ks=[k for k in ks if all(os.path.exists(f'{p}/{k:04d}.png') for _,p in cols)]
    fig,ax=plt.subplots(len(ks),len(cols),figsize=(1.9*len(cols),1.9*len(ks)))
    for i,k in enumerate(ks):
        for j,(t,p) in enumerate(cols):
            ax[i,j].imshow(Image.open(f'{p}/{k:04d}.png').convert('RGB').resize((256,256))); ax[i,j].axis('off')
            if i==0: ax[i,j].set_title(t,fontsize=9)
    plt.tight_layout(); plt.savefig(f'{D}/{name}_fixed.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('grid candidates ->',D,'| Vacher column:',('Vacher-$W_2$' in [c[0] for c in cols]))

In [ ]:
# APPS rebuild: transfer both ways + spatial ramp + 4-way (space-tolerant search, title band cropped)
D=cand_dir('apps')
hits=glob.glob(f'{BASE}/**/4*way*barrycenter*.png',recursive=True)+glob.glob(f'{BASE}/**/4*way*barycenter*.png',recursive=True)
FOURWAY=hits[0] if hits else None
print('4-way found at:',FOURWAY)
for v,(ka,kb) in enumerate([(12,317),(145,688),(45,555)]):
    A,B=pair_imgs(ka); d=build_d(cnt_load(A),cnt_load(B))
    tr_ab=to_pil(interp_ot(d,1.0,seed=ka))
    d_ba=build_d(cnt_load(B),cnt_load(A)); tr_ba=to_pil(interp_ot(d_ba,1.0,seed=ka))
    L,Cc,H,W,sf,tf=_grids(d); i_s=_sub(sf.shape[0],3,'s'); i_t=_sub(tf.shape[0],3,'t')
    P,_=sinkhorn_plan(sf[i_s],tf[i_t],EPS); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); ramp=torch.linspace(0,1,W,device=device).repeat(H,1).reshape(-1,1)
    sp=to_pil(_dec(d,(1-ramp)*sf+ramp*tt[nn],L,H,W,Cc))
    panels=[('A',A.resize((256,256))),('A layout, B material',tr_ab),('B layout, A material',tr_ba),
            ('B',B.resize((256,256))),('spatial η(x) ramp',sp)]
    if FOURWAY:
        fw=Image.open(FOURWAY).convert('RGB'); w,h=fw.size
        panels.append(('anchor-based 4-way',fw.crop((0,int(h*0.12),w,h))))
    fig,ax=plt.subplots(1,len(panels),figsize=(2.7*len(panels),2.8))
    for j,(t,im) in enumerate(panels):
        ax[j].imshow(im); ax[j].set_title(t,fontsize=8); ax[j].axis('off')
    plt.tight_layout(); plt.savefig(f'{D}/variant{v}_fixed.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('apps candidates ->',D)

In [ ]:
# BOUNDARY rerun: rotation 2 deg (so the correspondence rows match the text), faces row if provided
D=cand_dir('boundary'); FACES=f'{BASE}/faces'
def sift_count(a,b):
    try:
        s=cv2.SIFT_create(); g=lambda im: cv2.cvtColor(np.asarray(im.resize((256,256))),cv2.COLOR_RGB2GRAY)
        ka,da=s.detectAndCompute(g(a),None); kb,db=s.detectAndCompute(g(b),None)
        if da is None or db is None: return 0
        return sum(1 for p in cv2.BFMatcher().knnMatch(da,db,k=2) if len(p)==2 and p[0].distance<0.75*p[1].distance)
    except Exception: return -1
have_faces=os.path.exists(f'{FACES}/face_A.png') and os.path.exists(f'{FACES}/face_B.png')
for k in [12,145,317]:
    A,B=pair_imgs(k); W0,H0=A.size; rows=[]
    warp=A.rotate(2,resample=Image.BICUBIC).crop((12,12,W0-12,H0-12)).resize(A.size)
    big=A.resize((int(W0*1.4),int(H0*1.4)))
    c1=big.crop((0,0,W0,H0)); c2=big.crop((int(W0*.25),int(H0*.25),int(W0*.25)+W0,int(H0*.25)+H0))
    cases=[(A,warp,'small warp'),(c1,c2,'overlapping crops'),(A,B,'dissimilar')]
    if have_faces:
        cases.append((Image.open(f'{FACES}/face_A.png').convert('RGB'),
                      Image.open(f'{FACES}/face_B.png').convert('RGB'),'aligned faces'))
    for A2,B2,tag in cases:
        d=build_d(cnt_load(A2),cnt_load(B2))
        rows.append(([A2.resize((256,256)),to_pil(interp_lin_texton(d,.5)),to_pil(interp_ot(d,.5,seed=1)),B2.resize((256,256))],sift_count(A2,B2),tag))
    fig,ax=plt.subplots(len(rows),4,figsize=(10,2.5*len(rows)))
    for i,(ims_,cnt,tag) in enumerate(rows):
        for j,im in enumerate(ims_): ax[i,j].imshow(im); ax[i,j].axis('off')
        ax[i,0].text(-0.08,0.5,f'{tag}\nSIFT={cnt}',transform=ax[i,0].transAxes,fontsize=9,ha='right',va='center')
        if i==0:
            for j,t in enumerate(['A','linear','OT','B']): ax[i,j].set_title(t,fontsize=10)
    plt.tight_layout(); plt.savefig(f'{D}/base{k:04d}_r2.png',dpi=200,bbox_inches='tight'); plt.show(); plt.close()
print('boundary candidates ->',D,'| faces row:',have_faces)